# Solution D1 · Reading `cv_results_`: who really wins, and which parameter matters

**Type**: Interpret

## Objective

Analyze the results table of a `GridSearchCV` on a random forest: separate the truly better combinations from the
equivalent ones, apply the standard error rule to choose the simplest model and measure overfitting and the importance
of each hyperparameter.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

df = ds.loans(with_missing=False)
X = df[["age", "income", "job_tenure", "amount"]]
y = df["defaulted"]
grid = {"max_depth": [2, 4, 6, 10, None], "min_samples_leaf": [1, 5, 20]}
search = GridSearchCV(RandomForestClassifier(n_estimators=60, random_state=0), grid,
                      cv=StratifiedKFold(5, shuffle=True, random_state=0), scoring="roc_auc", return_train_score=True).fit(X, y)
r = pd.DataFrame(search.cv_results_)
r["max_depth"] = r["param_max_depth"].map(lambda v: 99 if v is None else v).astype(int)       # None = no limit
r["min_samples_leaf"] = r["param_min_samples_leaf"].astype(int)
r[["max_depth", "min_samples_leaf", "mean_test_score", "std_test_score", "mean_train_score", "rank_test_score"]].sort_values("rank_test_score").head(5).round(3)

## Commented solution

In [ ]:
r = r.sort_values("rank_test_score").reset_index(drop=True)
best = r.iloc[0]
best_mean, best_std = best["mean_test_score"], best["std_test_score"]
print(f"best: depth {best['max_depth']}, leaf {best['min_samples_leaf']} -> AUC {best_mean:.4f} +/- {best_std:.4f}")

In [ ]:
candidates_1se = r[r["mean_test_score"] >= best_mean - best_std]
n_candidates_1se = len(candidates_1se)
choice_1se = candidates_1se.sort_values(["max_depth", "min_samples_leaf"], ascending=[True, False]).iloc[0]
print(f"{n_candidates_1se} combinations out of {len(r)} are within one standard error of the best")
print(f"1-SE choice: depth {choice_1se['max_depth']}, leaf {choice_1se['min_samples_leaf']} -> AUC {choice_1se['mean_test_score']:.4f}")

In [ ]:
r["gap"] = r["mean_train_score"] - r["mean_test_score"]
best_gap = r.iloc[0]["gap"]
max_gap = r["gap"].max()
print(f"train-validation gap: best {best_gap:.3f} | maximum {max_gap:.3f}")

In [ ]:
m_depth = r.groupby("max_depth")["mean_test_score"].mean()
m_leaf = r.groupby("min_samples_leaf")["mean_test_score"].mean()
depth_range = m_depth.max() - m_depth.min()
leaf_range = m_leaf.max() - m_leaf.min()
most_influential_parameter = "max_depth" if depth_range > leaf_range else "min_samples_leaf"
print("mean AUC per max_depth:", m_depth.round(3).to_dict())
print("mean AUC per min_samples_leaf:", m_leaf.round(3).to_dict())
print("most influential:", most_influential_parameter)

## Check

In [ ]:
with verify():
    check(best["rank_test_score"] == 1 and close(best_mean, r["mean_test_score"].max()) and best_std > 0,
          "the best is the first row and has a positive standard deviation", "best = r.iloc[0]")
    expected = r[r["mean_test_score"] >= best_mean - best_std]
    check(n_candidates_1se == len(expected) and n_candidates_1se >= 5,
          "many combinations are statistically indistinguishable from the best (at least 5)", "candidates_1se = rows with mean_test_score >= best_mean - best_std")
    order = candidates_1se.assign(_d=candidates_1se["max_depth"]).sort_values(["_d", "min_samples_leaf"], ascending=[True, False])
    check(choice_1se["max_depth"] == order.iloc[0]["max_depth"] and choice_1se["min_samples_leaf"] == order.iloc[0]["min_samples_leaf"],
          "choice_1se: the simplest combination among the equivalent ones", "minimum max_depth (None = 99), on a tie maximum min_samples_leaf")
    check(choice_1se["max_depth"] <= best["max_depth"] and choice_1se["mean_test_score"] >= best_mean - best_std,
          "the choice is as simple as the best or more so, without losing appreciably", "the 1-SE choice cannot be more complex than the best")
    check(close(best_gap, best["mean_train_score"] - best["mean_test_score"]) and max_gap >= best_gap and max_gap > 0.2,
          "the maximum gap (overfitting) exceeds 0.2", "gap = mean_train_score - mean_test_score")
    check(most_influential_parameter in ("max_depth", "min_samples_leaf") and
          most_influential_parameter == ("max_depth" if depth_range > leaf_range else "min_samples_leaf"),
          "most_influential_parameter consistent with the ranges", "the parameter with the larger range of the means")
done()

## What you learned

- The standard deviation across folds is large compared to the differences between combinations: many (here most) are
  **statistically indistinguishable** from the best. The "winner" is little more than a random choice in the middle of a group of equals.
- The **standard error rule** picks among the equals the simplest model (shallower, more regularized): at equal
  estimated performance it usually generalizes better, and costs less to serve.
- The **gap** between training and validation shows where the model memorizes: the combination with no depth limit and leaves of a
  single example has a gap of 0.25 (training close to 1, validation much lower), against 0.07 for the best. A huge gap is not a bug, it is the
  signature of overfitting.
- Aggregating by value of each hyperparameter shows which one matters: here the maximum depth moves the mean by about 0.014 of AUC and
  `min_samples_leaf` by about 0.010. They are small differences, of the same order as the standard deviation across folds: among these
  hyperparameters none changes the performance dramatically.

## Common pitfall

Taking `best_params_` as a discovery ("the optimal depth is 6"). With the variability of the folds the "optimal" depth changes if
you change seed or sample: the correct message is "among these combinations there is no demonstrable difference".

## Interview question

> *Click on a question to show its answer.*

<details><summary>Your GridSearchCV says the best has AUC 0.812 with max_depth=8, but the second has 0.811 with max_depth=4. Which one do you send to production?</summary>

I look at the standard deviation across folds: if it is 0.03 the difference of 0.001 is noise, and I apply the standard error rule choosing
the simplest model among the indistinguishable ones: the one with max_depth=4 (less overfitting, faster, more stable). If I had
reasons to prefer the other (for example a different test set that confirms it) I would write them down, but "first in the ranking" is not a reason.

</details>